# RAE latent 群作用第一阶段验证

目标是验证固定空间置换 `P_g` 下，是否存在通道线性算子 `C_g`，使得 `E(gx) ≈ P_g E(x) C_g^T`。这个 notebook 只做实验调度、指标查看和少量可视化；重计算逻辑在 `experiments/group_action_phase1.py`。

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

import numpy as np
import pandas as pd
import torch
from IPython.display import display

CWD = Path.cwd().resolve()
ROOT = CWD if (CWD / "train_eqvae").exists() else CWD.parent
sys.path.insert(0, str(ROOT))

from experiments.group_action_phase1 import (
    HF_VAE_SPECS,
    RAE_SPECS,
    TRANSFORMS,
    ImageTensorDataset,
    Phase1NotebookSession,
    P as fixed_P,
    center_crop_resize,
    load_named_dataset,
    pil_to_tensor_m11,
    split_indices,
)

torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
print(f"ROOT = {ROOT}")
print(f"CUDA = {torch.cuda.is_available()}, GPUs = {torch.cuda.device_count()}")

## 1. 运行第一阶段实验

默认使用 4 张 GPU、fp32、固定 `P_g`，并对 `RAE-DINOv2 / RAE-MAE / RAE-SigLIP2 / EQVAE / SD-VAE` 依次拟合 `C_g`。第一次建议先把 `run_now` 设为 `False` 检查命令，再改成 `True` 执行。

In [ ]:
artifact_dir = "artifacts/group_action_phase1"
# 如果要看 DINOv2 生成元约束版本，可以改成对应 artifact：
# artifact_dir = "artifacts/group_action_phase1_generators_rae_dinov2"
gpu_ids = "0,1,2,3"
model_names = "rae_dinov2,rae_mae,rae_siglip2,eqvae,sdvae"

train_count = 512
val_count = 256
test_count = 256
batch_size = 16
num_workers = 4
seed = 0

extra_args = []
# 只对 rae_dinov2 训练生成元约束 C_g 时打开下面一行：
# extra_args = ["--fit-generators", "--generator-models", "rae_dinov2"]

cmd = [
    "torchrun", "--standalone", "--nproc_per_node=4",
    "experiments/group_action_phase1.py",
    "--models", model_names,
    "--data-root", "/data/shared",
    "--dataset-name", "caltech101",
    "--seed", str(seed),
    "--train-count", str(train_count),
    "--val-count", str(val_count),
    "--test-count", str(test_count),
    "--batch-size", str(batch_size),
    "--num-workers", str(num_workers),
    "--artifact-dir", artifact_dir,
]
cmd += extra_args

print("CUDA_VISIBLE_DEVICES=" + gpu_ids + " " + " ".join(cmd))
run_now = False
if run_now:
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": gpu_ids}
    subprocess.run(cmd, cwd=ROOT, env=env, check=True)

## 2. 查看指标

`Err_P` 是只做固定空间置换的误差，`Err_PC` 是再加通道算子后的误差。第一阶段主要看 `ratio_pc_over_p = Err_PC / Err_P` 和群律误差。

In [ ]:
summary_path = ROOT / artifact_dir / "summary.json"

def load_summary(path=summary_path):
    if not path.exists():
        print(f"还没有 summary: {path}")
        return None
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

def _metric_blocks(payload):
    blocks = [("independent", payload)]
    if "generator_constrained" in payload:
        blocks.append(("generator", payload["generator_constrained"]))
    return blocks

def metrics_table(summary, split="test"):
    rows = []
    for model_name, payload in summary.get("models", {}).items():
        for fit_name, block in _metric_blocks(payload):
            for transform, metrics in block["metrics"][split].items():
                rows.append({"model": model_name, "fit": fit_name, "transform": transform, **metrics})
    return pd.DataFrame(rows)

def group_law_table(summary):
    rows = []
    for model_name, payload in summary.get("models", {}).items():
        for fit_name, block in _metric_blocks(payload):
            rows.append({"model": model_name, "fit": fit_name, **block["group_law"], **block["success"]})
    return pd.DataFrame(rows)

summary = load_summary()
if summary is not None:
    display(metrics_table(summary, "test"))
    display(group_law_table(summary))

## 3. 简短操作接口

`P(z, g)` 是固定几何置换，不训练。`C(z, g, model_name)` 是已经拟合好的通道线性算子。`V_group(x, g, model_name)` 会显示 `x / g(x) / D(P_g z) / D(P_g z C_g^T) / D(E(gx))`。

In [ ]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
# 可选："independent" 或 "generator"。generator 会读取 *_generator_maps.pt。
fit_type = "independent"
session = Phase1NotebookSession(artifact_dir=artifact_dir, device=device, image_size=256, fit_type=fit_type)

def E_rae_dinov2(x): return session.E("rae_dinov2", x)
def D_rae_dinov2(z): return session.D("rae_dinov2", z)
def E_rae_mae(x): return session.E("rae_mae", x)
def D_rae_mae(z): return session.D("rae_mae", z)
def E_rae_siglip2(x): return session.E("rae_siglip2", x)
def D_rae_siglip2(z): return session.D("rae_siglip2", z)
def E_eqvae(x): return session.E("eqvae", x)
def D_eqvae(z): return session.D("eqvae", z)
def E_sdvae(x): return session.E("sdvae", x)
def D_sdvae(z): return session.D("sdvae", z)

def P(z, g="rot90"):
    return fixed_P(z, g)

def C(z, g="rot90", model_name="rae_dinov2"):
    return session.C(z, g, model_name=model_name)

def V_group(x, g="rot90", model_name="rae_dinov2", cell_size=192):
    grid = session.V_group(x, transform=g, model_name=model_name, cell_size=cell_size)
    display(grid)
    return grid

print("接口已就绪：E_rae_dinov2/D_rae_dinov2/E_eqvae/D_eqvae/E_sdvae/D_sdvae/P/C/V_group")

## 4. 从 Caltech101 选图

`pick_x` 返回 `[B,3,256,256]` 的 fp32 tensor，取值范围是 `[-1, 1]`。`positions` 是当前 split 内的位置；不传时用随机种子抽样。

In [ ]:
base_dataset = load_named_dataset("caltech101", "/data/shared", "train", download=False)
split_map = split_indices(len(base_dataset), train_count, val_count, test_count, seed)

def _tensor_from_original_index(index):
    sample = base_dataset[int(index)]
    img = sample[0] if isinstance(sample, (tuple, list)) else sample
    img = center_crop_resize(img.convert("RGB"), 256)
    return pil_to_tensor_m11(img)

def pick_x(split="test", positions=None, count=2, seed=0):
    pool = split_map[split]
    if positions is None:
        rng = np.random.default_rng(seed)
        positions = rng.choice(len(pool), size=min(count, len(pool)), replace=False)
    original_indices = [pool[int(pos)] for pos in positions]
    x = torch.stack([_tensor_from_original_index(idx) for idx in original_indices]).to(session.device)
    print({"split": split, "positions": [int(p) for p in positions], "original_indices": original_indices})
    return x

x = pick_x(split="test", count=2, seed=0)
x.shape

## 5. 可视化群作用

如果这里提示缺少 `*_maps.pt`，先回到第 1 节运行实验，或者把 `artifact_dir` 改成已有的实验目录。

In [ ]:
model_name = "rae_dinov2"
g = "rot90"

# 低层操作也可以直接拆开看：
z = E_rae_dinov2(x)
z_p = P(z, g)
z_pc = C(z_p, g, model_name=model_name)

V_group(x, g=g, model_name=model_name, cell_size=192)